# 🧠 Multi-head Latent Attention (MLA)

이 노트북에서는 DeepSeek V3의 핵심 혁신인 **Multi-head Latent Attention (MLA)**를 학습합니다.

MLA는 KV 캐시를 저차원 잠재 벡터로 압축하여 메모리 효율성을 크게 향상시킵니다.

**참고 자료:**
- 논문: https://arxiv.org/pdf/2412.19437

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 1. MLA 핵심 개념

### 기존 MHA의 문제점
- KV 캐시 크기: `2 × n_layers × n_heads × head_dim × seq_len × batch`
- 긴 시퀀스에서 메모리 사용량이 급격히 증가

### MLA의 해결책
- KV를 저차원 잠재 벡터 `c_KV`로 압축하여 저장
- 추론 시 `c_KV`에서 K, V를 복원
- 캐시 크기 대폭 감소!

### 핵심 수식

**1. KV 압축 (Down Projection):**
$$c_{KV} = W_{DKV} \cdot h_t$$

**2. KV 복원 (Up Projection):**
$$k_C = W_{UK} \cdot c_{KV}$$
$$v = W_{UV} \cdot c_{KV}$$

**3. Decoupled RoPE:**
$$k = [k_C; \text{RoPE}(k_R)]$$
$$q = [q_C; \text{RoPE}(q_R)]$$

In [2]:
# MLA 간소화 구현
class SimpleMLA(nn.Module):
    def __init__(self, hidden_size=512, num_heads=8, kv_lora_rank=128, head_dim=64):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.kv_lora_rank = kv_lora_rank
        self.head_dim = head_dim
        
        # Query projection
        self.q_proj = nn.Linear(hidden_size, num_heads * head_dim, bias=False)
        
        # KV compression (핵심!)
        self.kv_down_proj = nn.Linear(hidden_size, kv_lora_rank, bias=False)  # 압축
        self.k_up_proj = nn.Linear(kv_lora_rank, num_heads * head_dim, bias=False)  # K 복원
        self.v_up_proj = nn.Linear(kv_lora_rank, num_heads * head_dim, bias=False)  # V 복원
        
        # Output projection
        self.o_proj = nn.Linear(num_heads * head_dim, hidden_size, bias=False)
    
    def forward(self, hidden_states):
        batch_size, seq_len, _ = hidden_states.shape
        
        # Query
        q = self.q_proj(hidden_states)
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # KV compression and recovery
        kv_compressed = self.kv_down_proj(hidden_states)  # 압축!
        k = self.k_up_proj(kv_compressed)
        v = self.v_up_proj(kv_compressed)
        
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention
        scale = 1.0 / math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        # Output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.o_proj(attn_output), kv_compressed

print("✅ SimpleMLA 구현 완료")

✅ SimpleMLA 구현 완료


## 📝 이해도 테스트 1: MLA 동작 확인

In [3]:
# MLA 동작 테스트
hidden_size = 512
num_heads = 8
kv_lora_rank = 128
head_dim = 64

mla = SimpleMLA(hidden_size, num_heads, kv_lora_rank, head_dim)

batch_size, seq_len = 2, 16
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

output, kv_compressed = mla(hidden_states)

print("📊 MLA 동작 테스트")
print("=" * 50)
print(f"입력 형태: {hidden_states.shape}")
print(f"출력 형태: {output.shape}")
print(f"\n💡 핵심: KV 압축 벡터")
print(f"   kv_compressed 형태: {kv_compressed.shape}")
print(f"   압축 전 크기: {num_heads * head_dim * 2} (K + V)")
print(f"   압축 후 크기: {kv_lora_rank}")
print(f"   압축률: {kv_lora_rank / (num_heads * head_dim * 2) * 100:.1f}%")

📊 MLA 동작 테스트
입력 형태: torch.Size([2, 16, 512])
출력 형태: torch.Size([2, 16, 512])

💡 핵심: KV 압축 벡터
   kv_compressed 형태: torch.Size([2, 16, 128])
   압축 전 크기: 1024 (K + V)
   압축 후 크기: 128
   압축률: 12.5%


## 2. 메모리 효율성 분석

In [4]:
# 메모리 효율성 분석
def analyze_memory(n_layers, n_heads, head_dim, kv_lora_rank, seq_len, batch_size=1):
    bytes_per_param = 2  # FP16
    
    # 기존 MHA KV 캐시
    mha_kv_cache = 2 * n_layers * batch_size * n_heads * seq_len * head_dim * bytes_per_param
    
    # MLA KV 캐시 (압축된 c_KV만 저장)
    mla_kv_cache = n_layers * batch_size * seq_len * kv_lora_rank * bytes_per_param
    
    return mha_kv_cache / (1024**3), mla_kv_cache / (1024**3)

# DeepSeek V3 스케일
n_layers = 61
n_heads = 128
head_dim = 128
kv_lora_rank = 512

print("📊 KV 캐시 메모리 비교 (DeepSeek V3 스케일)")
print("=" * 60)
print(f"{'시퀀스 길이':>12} | {'MHA (GB)':>10} | {'MLA (GB)':>10} | {'절감률':>8}")
print("-" * 60)

for seq_len in [1024, 4096, 16384, 65536, 131072]:
    mha, mla = analyze_memory(n_layers, n_heads, head_dim, kv_lora_rank, seq_len)
    reduction = (1 - mla/mha) * 100
    print(f"{seq_len:>12,} | {mha:>10.2f} | {mla:>10.2f} | {reduction:>7.1f}%")

📊 KV 캐시 메모리 비교 (DeepSeek V3 스케일)
      시퀀스 길이 |   MHA (GB) |   MLA (GB) |      절감률
------------------------------------------------------------
       1,024 |       3.81 |       0.06 |    98.4%
       4,096 |      15.25 |       0.24 |    98.4%
      16,384 |      61.00 |       0.95 |    98.4%
      65,536 |     244.00 |       3.81 |    98.4%
     131,072 |     488.00 |       7.62 |    98.4%


## 📝 이해도 테스트 2: 압축률 계산

아래 질문에 답해보세요.

In [5]:
# 퀴즈: 압축률 계산

# 설정
hidden_size = 7168
n_heads = 128
head_dim = 128
kv_lora_rank = 512

# 질문: 기존 MHA에서 K+V의 총 차원은?
# 힌트: n_heads * head_dim * 2
mha_kv_dim = None  # TODO: 계산하세요

# 질문: MLA에서 압축된 c_KV의 차원은?
mla_kv_dim = None  # TODO

# 질문: 압축률은?
compression_ratio = None  # TODO: mla_kv_dim / mha_kv_dim

print("빈칸을 채우고 실행하세요!")

빈칸을 채우고 실행하세요!


In [6]:
# 정답
mha_kv_dim = n_heads * head_dim * 2
print(f"기존 MHA K+V 차원: {mha_kv_dim:,}")

mla_kv_dim = kv_lora_rank
print(f"MLA c_KV 차원: {mla_kv_dim:,}")

compression_ratio = mla_kv_dim / mha_kv_dim
print(f"\n압축률: {compression_ratio*100:.2f}%")
print(f"메모리 절감: {(1-compression_ratio)*100:.2f}%")

기존 MHA K+V 차원: 32,768
MLA c_KV 차원: 512

압축률: 1.56%
메모리 절감: 98.44%


## 3. Decoupled RoPE 이해

MLA에서 RoPE를 왜 분리해서 적용할까요?

### 문제점
- 압축된 `c_KV`에서 복원한 K에 RoPE를 적용하면 위치 정보가 손실될 수 있음

### 해결책: Decoupled RoPE
- `k_nope`: 압축/복원되는 부분 (위치 무관 정보)
- `k_rope`: 별도로 계산되는 RoPE 부분 (위치 정보)
- 최종 키 = `[k_nope; k_rope]`

In [7]:
# MLA 데이터 흐름 시각화
print("""
┌─────────────────────────────────────────────────────────────┐
│                Multi-head Latent Attention (MLA)            │
└─────────────────────────────────────────────────────────────┘

                          Input: h_t
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
        ▼                    ▼                    ▼
   ┌─────────┐         ┌─────────┐         ┌─────────┐
   │ Q proj  │         │ KV down │         │ K_rope  │
   └────┬────┘         └────┬────┘         └────┬────┘
        │                   │ c_KV              │
        │                   │ [저장됨!]          │
        │           ┌───────┴───────┐           │
        │           ▼               ▼           │
        │      ┌─────────┐    ┌─────────┐      │
        │      │ K up    │    │ V up    │      │
        │      └────┬────┘    └────┬────┘      │
        │           │ k_nope       │           │
        │           ▼              │           │
        │      ┌─────────┐         │           │
        │      │ Concat  │◄────────┴───────────┘
        │      │k_nope,  │         │
        │      │k_rope   │         │
        │      └────┬────┘         │
        │           │ K            │ V
        ▼           ▼              ▼
   ┌─────────────────────────────────────────────┐
   │      Attention(Q, K, V) = softmax(QK^T/√d)V │
   └─────────────────────────────────────────────┘
""")

print("💡 핵심 포인트:")
print("   - c_KV만 캐시에 저장 (메모리 절약!)")
print("   - K, V는 추론 시 c_KV에서 복원")
print("   - RoPE는 별도로 적용 (위치 정보 보존)")


┌─────────────────────────────────────────────────────────────┐
│                Multi-head Latent Attention (MLA)            │
└─────────────────────────────────────────────────────────────┘

                          Input: h_t
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
        ▼                    ▼                    ▼
   ┌─────────┐         ┌─────────┐         ┌─────────┐
   │ Q proj  │         │ KV down │         │ K_rope  │
   └────┬────┘         └────┬────┘         └────┬────┘
        │                   │ c_KV              │
        │                   │ [저장됨!]          │
        │           ┌───────┴───────┐           │
        │           ▼               ▼           │
        │      ┌─────────┐    ┌─────────┐      │
        │      │ K up    │    │ V up    │      │
        │      └────┬────┘    └────┬────┘      │
        │           │ k_nope       │           │
        │           ▼  

In [11]:
# k_nope와 k_rope concat 예시
print("🔧 k_nope와 k_rope concat 동작 예시")
print("=" * 60)

# 설정
batch_size = 2
seq_len = 4
num_heads = 8
head_dim_nope = 48  # 압축/복원되는 부분 (위치 무관)
head_dim_rope = 16  # RoPE가 적용되는 부분 (위치 정보)
total_head_dim = head_dim_nope + head_dim_rope  # 64

print(f"\n📐 차원 설정:")
print(f"   head_dim_nope (위치 무관): {head_dim_nope}")
print(f"   head_dim_rope (위치 정보): {head_dim_rope}")
print(f"   total_head_dim: {total_head_dim}")

# k_nope: c_KV에서 복원된 키 (압축 경로)
# 실제로는 W_UK @ c_KV로 계산됨
k_nope = torch.randn(batch_size, num_heads, seq_len, head_dim_nope)
print(f"\n1️⃣ k_nope (압축에서 복원):")
print(f"   shape: {k_nope.shape}")
print(f"   의미: c_KV → W_UK → k_nope")

# k_rope: 별도로 계산된 RoPE 적용 키
# 실제로는 RoPE(W_KR @ h_t)로 계산됨
k_rope = torch.randn(batch_size, num_heads, seq_len, head_dim_rope)
print(f"\n2️⃣ k_rope (별도 계산 + RoPE):")
print(f"   shape: {k_rope.shape}")
print(f"   의미: h_t → W_KR → RoPE → k_rope")

# Concat: 최종 키 생성
# dim=-1로 마지막 차원(head_dim)을 따라 concat
k_final = torch.cat([k_nope, k_rope], dim=-1)
print(f"\n3️⃣ 최종 K = concat(k_nope, k_rope):")
print(f"   shape: {k_final.shape}")
print(f"   {head_dim_nope} + {head_dim_rope} = {total_head_dim}")

# 실제 concat 동작 확인
print("\n" + "=" * 60)
print("🔍 실제 concat 동작 검증:")
print(f"   k_final[..., :48] == k_nope: {torch.allclose(k_final[..., :head_dim_nope], k_nope)}")
print(f"   k_final[..., 48:] == k_rope: {torch.allclose(k_final[..., head_dim_nope:], k_rope)}")


🔧 k_nope와 k_rope concat 동작 예시

📐 차원 설정:
   head_dim_nope (위치 무관): 48
   head_dim_rope (위치 정보): 16
   total_head_dim: 64

1️⃣ k_nope (압축에서 복원):
   shape: torch.Size([2, 8, 4, 48])
   의미: c_KV → W_UK → k_nope

2️⃣ k_rope (별도 계산 + RoPE):
   shape: torch.Size([2, 8, 4, 16])
   의미: h_t → W_KR → RoPE → k_rope

3️⃣ 최종 K = concat(k_nope, k_rope):
   shape: torch.Size([2, 8, 4, 64])
   48 + 16 = 64

🔍 실제 concat 동작 검증:
   k_final[..., :48] == k_nope: True
   k_final[..., 48:] == k_rope: True


In [12]:
# Q도 마찬가지로 concat! (q_nope + q_rope)
print("🔧 Query도 동일하게 concat 적용")
print("=" * 60)

# Query도 nope + rope 구조
q_nope = torch.randn(batch_size, num_heads, seq_len, head_dim_nope)
q_rope = torch.randn(batch_size, num_heads, seq_len, head_dim_rope)
q_final = torch.cat([q_nope, q_rope], dim=-1)

print(f"q_nope shape: {q_nope.shape}")
print(f"q_rope shape: {q_rope.shape}")
print(f"q_final shape: {q_final.shape}")

# Attention 계산 시 Q @ K^T에서 어떻게 동작하는지
print("\n" + "=" * 60)
print("💡 Attention에서의 의미:")
print("""
Q @ K^T = [q_nope; q_rope] @ [k_nope; k_rope]^T
        = q_nope @ k_nope^T + q_rope @ k_rope^T
        
→ nope 부분: 의미적 유사도 (content-based)
→ rope 부분: 위치 기반 유사도 (position-based)
""")

# 실제 계산 예시
attn_from_nope = torch.matmul(q_nope, k_nope.transpose(-2, -1))
attn_from_rope = torch.matmul(q_rope, k_rope.transpose(-2, -1))
attn_total = torch.matmul(q_final, k_final.transpose(-2, -1))

print(f"nope 기여 shape: {attn_from_nope.shape}")
print(f"rope 기여 shape: {attn_from_rope.shape}")
print(f"총 attention shape: {attn_total.shape}")
print(f"\n검증 (nope + rope == total): {torch.allclose(attn_from_nope + attn_from_rope, attn_total)}")


🔧 Query도 동일하게 concat 적용
q_nope shape: torch.Size([2, 8, 4, 48])
q_rope shape: torch.Size([2, 8, 4, 16])
q_final shape: torch.Size([2, 8, 4, 64])

💡 Attention에서의 의미:

Q @ K^T = [q_nope; q_rope] @ [k_nope; k_rope]^T
        = q_nope @ k_nope^T + q_rope @ k_rope^T
        
→ nope 부분: 의미적 유사도 (content-based)
→ rope 부분: 위치 기반 유사도 (position-based)

nope 기여 shape: torch.Size([2, 8, 4, 4])
rope 기여 shape: torch.Size([2, 8, 4, 4])
총 attention shape: torch.Size([2, 8, 4, 4])

검증 (nope + rope == total): False


## 3.1 Decoupled RoPE가 포함된 완전한 MLA 구현

위에서 배운 `k_nope`, `k_rope` concat 개념을 실제 MLA 클래스로 구현합니다.


In [13]:
# Decoupled RoPE가 포함된 완전한 MLA 구현
class MLAWithDecoupledRoPE(nn.Module):
    """
    Multi-head Latent Attention with Decoupled RoPE
    
    핵심 구조:
    - Q = [q_nope; q_rope]  (nope: 압축 경로, rope: 위치 정보)
    - K = [k_nope; k_rope]  (nope: c_KV에서 복원, rope: 별도 계산)
    - V = v_nope            (c_KV에서 복원)
    """
    def __init__(
        self,
        hidden_size=512,
        num_heads=8,
        kv_lora_rank=128,
        head_dim_nope=48,    # 압축되는 부분의 차원
        head_dim_rope=16,    # RoPE 적용 부분의 차원
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.kv_lora_rank = kv_lora_rank
        self.head_dim_nope = head_dim_nope
        self.head_dim_rope = head_dim_rope
        self.head_dim = head_dim_nope + head_dim_rope  # 총 head 차원
        
        # ========== Query 관련 ==========
        # q_nope: 압축 경로와 매칭되는 부분
        self.q_nope_proj = nn.Linear(hidden_size, num_heads * head_dim_nope, bias=False)
        # q_rope: RoPE가 적용될 부분
        self.q_rope_proj = nn.Linear(hidden_size, num_heads * head_dim_rope, bias=False)
        
        # ========== KV 압축 (핵심!) ==========
        # Down projection: h_t → c_KV (압축)
        self.kv_down_proj = nn.Linear(hidden_size, kv_lora_rank, bias=False)
        
        # Up projection: c_KV → k_nope, v (복원)
        self.k_nope_up_proj = nn.Linear(kv_lora_rank, num_heads * head_dim_nope, bias=False)
        self.v_up_proj = nn.Linear(kv_lora_rank, num_heads * head_dim_nope, bias=False)
        
        # ========== k_rope (별도 경로) ==========
        # RoPE용 키는 압축 없이 직접 계산
        self.k_rope_proj = nn.Linear(hidden_size, num_heads * head_dim_rope, bias=False)
        
        # Output projection
        self.o_proj = nn.Linear(num_heads * head_dim_nope, hidden_size, bias=False)
        
    def apply_rope(self, x, seq_len):
        """간단한 RoPE 적용 (실제로는 더 복잡함)"""
        # 여기서는 개념 설명을 위해 단순화된 RoPE 사용
        device = x.device
        position = torch.arange(seq_len, device=device).unsqueeze(0)
        dim = x.shape[-1]
        
        # 주파수 계산
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, device=device).float() / dim))
        
        # 위치 인코딩
        freqs = torch.einsum('bi,j->bij', position.float(), inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        
        cos = emb.cos().unsqueeze(1)  # [1, 1, seq_len, dim]
        sin = emb.sin().unsqueeze(1)
        
        # Rotate half
        x1, x2 = x[..., :dim//2], x[..., dim//2:]
        rotated = torch.cat([-x2, x1], dim=-1)
        
        return x * cos + rotated * sin
    
    def forward(self, hidden_states):
        batch_size, seq_len, _ = hidden_states.shape
        
        # ========== Query 계산 ==========
        # q_nope: 압축 경로와 매칭
        q_nope = self.q_nope_proj(hidden_states)
        q_nope = q_nope.view(batch_size, seq_len, self.num_heads, self.head_dim_nope)
        q_nope = q_nope.transpose(1, 2)  # [B, H, S, D_nope]
        
        # q_rope: RoPE 적용
        q_rope = self.q_rope_proj(hidden_states)
        q_rope = q_rope.view(batch_size, seq_len, self.num_heads, self.head_dim_rope)
        q_rope = q_rope.transpose(1, 2)  # [B, H, S, D_rope]
        q_rope = self.apply_rope(q_rope, seq_len)  # RoPE 적용!
        
        # ========== KV 압축 및 복원 ==========
        # Step 1: 압축 (h_t → c_KV)
        c_kv = self.kv_down_proj(hidden_states)  # [B, S, kv_lora_rank]
        
        # Step 2: k_nope 복원 (c_KV → k_nope)
        k_nope = self.k_nope_up_proj(c_kv)
        k_nope = k_nope.view(batch_size, seq_len, self.num_heads, self.head_dim_nope)
        k_nope = k_nope.transpose(1, 2)  # [B, H, S, D_nope]
        
        # Step 3: k_rope 계산 (별도 경로, 압축 안 함!)
        k_rope = self.k_rope_proj(hidden_states)
        k_rope = k_rope.view(batch_size, seq_len, self.num_heads, self.head_dim_rope)
        k_rope = k_rope.transpose(1, 2)  # [B, H, S, D_rope]
        k_rope = self.apply_rope(k_rope, seq_len)  # RoPE 적용!
        
        # Step 4: V 복원 (c_KV → v)
        v = self.v_up_proj(c_kv)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim_nope)
        v = v.transpose(1, 2)  # [B, H, S, D_nope]
        
        # ========== Concat: 최종 Q, K 생성 ==========
        q = torch.cat([q_nope, q_rope], dim=-1)  # [B, H, S, D_nope + D_rope]
        k = torch.cat([k_nope, k_rope], dim=-1)  # [B, H, S, D_nope + D_rope]
        
        # ========== Attention 계산 ==========
        scale = 1.0 / math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        
        # V는 nope 차원만 사용 (head_dim_nope)
        attn_output = torch.matmul(attn_weights, v)  # [B, H, S, D_nope]
        
        # Output
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, -1)
        
        return self.o_proj(attn_output), c_kv

print("✅ MLAWithDecoupledRoPE 구현 완료")


✅ MLAWithDecoupledRoPE 구현 완료


In [14]:
# MLAWithDecoupledRoPE 테스트
print("🧪 MLAWithDecoupledRoPE 동작 테스트")
print("=" * 60)

# 설정
hidden_size = 512
num_heads = 8
kv_lora_rank = 128
head_dim_nope = 48
head_dim_rope = 16

mla_rope = MLAWithDecoupledRoPE(
    hidden_size=hidden_size,
    num_heads=num_heads,
    kv_lora_rank=kv_lora_rank,
    head_dim_nope=head_dim_nope,
    head_dim_rope=head_dim_rope,
)

# 입력 생성
batch_size, seq_len = 2, 16
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

# Forward pass
output, c_kv = mla_rope(hidden_states)

print(f"\n📊 입출력 형태:")
print(f"   입력 (hidden_states): {hidden_states.shape}")
print(f"   출력 (output): {output.shape}")
print(f"   압축된 KV (c_kv): {c_kv.shape}")

print(f"\n📐 차원 분석:")
print(f"   head_dim_nope: {head_dim_nope}")
print(f"   head_dim_rope: {head_dim_rope}")
print(f"   total head_dim: {head_dim_nope + head_dim_rope}")

print(f"\n💾 KV 캐시 저장량 비교:")
original_kv_size = num_heads * (head_dim_nope + head_dim_rope) * 2  # K + V
mla_cache_size = kv_lora_rank + num_heads * head_dim_rope  # c_kv + k_rope
print(f"   기존 MHA (K+V): {original_kv_size}")
print(f"   MLA (c_kv + k_rope): {mla_cache_size}")
print(f"   절감률: {(1 - mla_cache_size/original_kv_size)*100:.1f}%")

print(f"\n✅ 테스트 통과!")


🧪 MLAWithDecoupledRoPE 동작 테스트

📊 입출력 형태:
   입력 (hidden_states): torch.Size([2, 16, 512])
   출력 (output): torch.Size([2, 16, 512])
   압축된 KV (c_kv): torch.Size([2, 16, 128])

📐 차원 분석:
   head_dim_nope: 48
   head_dim_rope: 16
   total head_dim: 64

💾 KV 캐시 저장량 비교:
   기존 MHA (K+V): 1024
   MLA (c_kv + k_rope): 256
   절감률: 75.0%

✅ 테스트 통과!


## 4. 요약 퀴즈

In [8]:
quiz = {
    "Q1: MLA는 KV 캐시를 압축하여 저장한다": None,
    "Q2: MLA에서 K와 V는 항상 따로 압축된다": None,
    "Q3: Decoupled RoPE는 위치 정보 손실을 방지한다": None,
    "Q4: MLA는 기존 MHA보다 메모리를 더 많이 사용한다": None,
}

# 여기에 답을 입력하세요 (True or False)

In [9]:
answers = {
    "Q1: MLA는 KV 캐시를 압축하여 저장한다": True,
    "Q2: MLA에서 K와 V는 항상 따로 압축된다": False,  # 함께 압축됨 (Joint KV Compression)
    "Q3: Decoupled RoPE는 위치 정보 손실을 방지한다": True,
    "Q4: MLA는 기존 MHA보다 메모리를 더 많이 사용한다": False,
}

print("📋 퀴즈 정답")
print("=" * 50)
for q, a in answers.items():
    user_ans = quiz.get(q)
    status = "✅" if user_ans == a else "❌"
    print(f"{status} {q}")
    print(f"   정답: {a}\n")

📋 퀴즈 정답
❌ Q1: MLA는 KV 캐시를 압축하여 저장한다
   정답: True

❌ Q2: MLA에서 K와 V는 항상 따로 압축된다
   정답: False

❌ Q3: Decoupled RoPE는 위치 정보 손실을 방지한다
   정답: True

❌ Q4: MLA는 기존 MHA보다 메모리를 더 많이 사용한다
   정답: False

